In [1]:
import os
import json
from dotenv import load_dotenv
from utils import set_seed, generate_namespace
from transformers import Trainer, TrainingArguments

from lora_fine_tune.model_local import LocalLM
from lora_fine_tune.lora_utils import apply_lora
from lora_fine_tune.data_utils import build_dataset

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
load_dotenv()
api_key = os.environ["OPENAI_API_KEY"]

cfg = generate_namespace(path=f"../config.yaml")
print(json.dumps(vars(cfg), indent=2))

set_seed(cfg.seed)

{
  "model_name": "gpt2",
  "api_model_name": "gpt-4.1-nano",
  "seed": 42,
  "max_token_length": 128,
  "lr": 2e-05,
  "warmup_ratio": 0.1,
  "epochs": 5,
  "train_batch_size": 1,
  "strategy": "epoch",
  "res_path": "../results/"
}


In [3]:
model = LocalLM(cfg.model_name)

total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 124,439,808
Trainable parameters: 124,439,808


In [4]:
lora_model = apply_lora(model.model)
lora_model.print_trainable_parameters()

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


/opt/miniconda3/envs/lora-fine-tune-env/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [5]:
dataset = build_dataset()
data_tokenized = model.tokenize_dataset(dataset, cfg.max_token_length, padding=False)
data_collator = model.data_collator()
print(data_tokenized)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 3
})


In [6]:
training_args = TrainingArguments(
    output_dir=cfg.res_path,
    overwrite_output_dir=True,
    report_to="none",
    save_strategy=cfg.strategy,
    logging_strategy= cfg.strategy,
    learning_rate=cfg.lr,
    per_device_train_batch_size=cfg.train_batch_size,
    num_train_epochs=cfg.epochs,
    warmup_ratio=cfg.warmup_ratio,
    seed=cfg.seed
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=data_tokenized,
    data_collator=data_collator
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
3,4.930900
6,5.041500
9,4.916600
12,5.004600
15,4.632600


TrainOutput(global_step=15, training_loss=4.905256779988607, metrics={'train_runtime': 7.5528, 'train_samples_per_second': 1.986, 'train_steps_per_second': 1.986, 'total_flos': 69134238720.0, 'train_loss': 4.905256779988607, 'epoch': 5.0})